## Setups

In [2]:
import pandas as pd                                        # pandas: tables (DataFrames)
import plotly.express as px                                # Plotly Express: quick interactive charts
from sklearn.model_selection import train_test_split       # hides part of the data for honest testing
from sklearn.linear_model import LinearRegression, LogisticRegression   # regression (numbers) / classification (yes-no)
from sklearn.metrics import (mean_absolute_error, r2_score,             # regression scorecards
                             accuracy_score, confusion_matrix)          # classification scorecards

## Part 1 

### Exercise 1.1

In [6]:
df = pd.read_csv("week10_students.csv")        # load the CSV that sits next to this notebook
print("Shape (rows, columns):", df.shape)       # should be (150, 9)

features = ["quiz_avg", "assignments_avg", "attendance_pct"]      # the three inputs we will use

# correlation of score with each feature: -1 to +1 (near +1 = rise together, near 0 = no straight-line link)
corr = df[features + ["score"]].corr()["score"].drop("score")     # .drop removes score-vs-score (always 1)
print(corr.round(3))



Shape (rows, columns): (150, 9)
quiz_avg           0.801
assignments_avg    0.536
attendance_pct     0.238
Name: score, dtype: float64


### Exercise 1.2 

In [8]:
X = df[features]       
y = df["score"]       

# split: 75% of students to learn from (train), 25% hidden for testing
# random_state=42 fixes the "random" shuffle so everybody gets the same split (and the same checkpoints)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Train size:", len(X_train), "| Test size:", len(X_test))   



Train size: 112 | Test size: 38


## Part 2 

### Exercise 2.1 

In [9]:
reg = LinearRegression()          # create an empty regression model
reg.fit(X_train, y_train)         # "learn": find the coefficients that fit the TRAINING students best

for name, coef in zip(features, reg.coef_):     # pair each feature name with its coefficient
    print(f"{name:16s} coefficient = {coef:.3f}")
print(f"{'intercept':16s} = {reg.intercept_:.3f}")


quiz_avg         coefficient = 0.579
assignments_avg  coefficient = 0.365
attendance_pct   coefficient = 0.113
intercept        = -4.428


### Exercise 2.2 

In [10]:
pred = reg.predict(X_test)                       # predict scores for the 38 HIDDEN students
mae = mean_absolute_error(y_test, pred)          # MAE = average size of the mistakes, in exam points
r2 = r2_score(y_test, pred)                      # R2 = share of the differences between students explained (0 to 1)

print(f"MAE = {mae:.2f} points")                 
print(f"R2  = {r2:.2f}")                      


MAE = 3.33 points
R2  = 0.89


### Exercise 2.3 

In [11]:
# a one-row DataFrame with the SAME column names the model was trained on (name matching matters!)
new_student = pd.DataFrame([[70, 75, 90]], columns=features)    # quiz 70, assignments 75, attendance 90
print("New student predicted score:", round(reg.predict(new_student)[0], 1))


my_student = pd.DataFrame([[65, 60, 80]], columns=features)     
print("My invented student predicted score:", round(reg.predict(my_student)[0], 1))

New student predicted score: 73.7
My invented student predicted score: 64.2


## Part 3 

### Exercise 3.1 

In [12]:
df["passed"] = (df["score"] >= 50).astype(int)      # True/False -> 1 (pass) / 0 (fail)
print(df["passed"].value_counts())                   # how many passed (1) and failed (0)?

# split again with the same settings, but this time the target is "passed" (yc = y for classification)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, df["passed"], test_size=0.25, random_state=42)

clf = LogisticRegression(max_iter=1000)    # max_iter=1000 gives it enough steps to finish learning
clf.fit(Xc_train, yc_train)               

passed
1    131
0     19
Name: count, dtype: int64


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

### Exercise 3.2 

In [13]:
pred_c = clf.predict(Xc_test)                                  # predict pass/fail for the hidden students
acc = accuracy_score(yc_test, pred_c)                          # share of correct predictions
baseline = yc_test.mean()                                      # lazy model "everyone passes" is right this often
print(f"Model accuracy    : {acc:.3f}")
print(f"Lazy baseline     : {baseline:.3f}")

cm = confusion_matrix(yc_test, pred_c)                         # rows = real (fail, pass), columns = predicted
print("Confusion matrix:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"Real fails caught: {tn} of {tn + fp} | passing students wrongly flagged: {fn}")


Model accuracy    : 0.895
Lazy baseline     : 0.895
Confusion matrix:
[[ 3  1]
 [ 3 31]]
Real fails caught: 3 of 4 | passing students wrongly flagged: 3


## Part 4 

### Exercise 4.1 

In [14]:
def test_r2(features_table):
    # split with the usual settings, train a fresh regression, and return R2 on the TEST students
    a_train, a_test, b_train, b_test = train_test_split(features_table, df["score"], test_size=0.25, random_state=42)
    model = LinearRegression().fit(a_train, b_train)
    return r2_score(b_test, model.predict(a_test))

program_dummies = pd.get_dummies(df["program"], dtype=int)     # one-hot: one 0/1 column per program

results = pd.DataFrame({
    "Version": ["1. quiz_avg only", "2. all three numeric features", "3. all three + program (one-hot)"],
    "Test R2": [test_r2(df[["quiz_avg"]]),
                test_r2(df[features]),
                test_r2(pd.concat([df[features], program_dummies], axis=1))],
})
results["Test R2"] = results["Test R2"].round(2)
print(results.to_string(index=False))        # checkpoint: roughly 0.62 -> 0.89 -> 0.88



                         Version  Test R2
                1. quiz_avg only     0.62
   2. all three numeric features     0.89
3. all three + program (one-hot)     0.88


## Part 5 

### Exercise 5.1 

In [10]:
# (a) histogram: how are scores spread? nbins=12 -> 12 bars
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

In [11]:
# (b) scatter: quiz average vs score, coloured by pass/fail, hover shows the student's name
df["status"] = df["passed"].map({1: "Passed (1)", 0: "At risk (0)"})
fig2 = px.scatter(df, x="quiz_avg", y="score", color="status", hover_name="name",
                  color_discrete_map={"Passed (1)": "#1FA08E", "At risk (0)": "#E67E22"},
                  labels={"quiz_avg": "Quiz average", "score": "Exam score", "status": "passed"},
                  title="2. Who is at risk")
fig2.add_hline(y=50, line_dash="dash", annotation_text="pass mark (50)")   # reference line at the pass mark
fig2.show()

In [12]:
# (c) bar chart: average score per district (sorted so the lowest is on the left)
district_avg = df.groupby("district", as_index=False)["score"].mean().sort_values("score")
fig3 = px.bar(district_avg, x="district", y="score", title="3. Where to act")
fig3.update_yaxes(title="Average score")
fig3.show()
print(district_avg.round(1).to_string(index=False))   # Gasabo is lowest (about 59), Kicukiro highest (about 68)

  district  score
    Gasabo   59.2
 Nyagatare   63.7
   Muhanga   63.7
      Huye   64.0
    Rubavu   65.0
Nyarugenge   65.1
   Musanze   66.3
  Kicukiro   68.3


### Exercise 5.2

In [13]:
fig2.write_html("class_dashboard.html")    # saves an interactive web page: open it in a browser and hover over dots
print("Saved class_dashboard.html")        # then take a screenshot of it for your submission

Saved class_dashboard.html


### Bonus 


In [14]:
fig4 = px.box(df, x="program", y="score", color="program", title="Score spread by program")
fig4.show()
print(df.groupby("program")["score"].agg(["mean", "std", "min", "max"]).round(1))   # the numbers behind the boxes

                      mean   std   min   max
program                                     
Accounting            66.7  11.0  43.0  89.5
Information Systems   61.6  12.3  28.6  84.4
Networking            64.4  10.8  41.9  89.4
Software Engineering  65.0  13.8  37.2  92.9


**Bonus answer (two sentences):** **Networking** is the most consistent program: it has the smallest standard deviation
(about 10.8) and the narrowest box (middle 50% spans about 14 points). **Software Engineering** has the widest spread
(standard deviation about 13.8 and scores from about 37 to 93), so it contains both the lowest and the highest performers.

## Part 6 · The Model Card

**1. Purpose.** This model predicts a student's final exam score (0–100) from their quiz average, assignment average and
attendance percentage. It is meant for lecturers and advisers at AUCA, to spot students who may need extra support
early in the semester.

**2. Data & features.** It was trained on 150 AUCA students (clean data: no missing values or duplicates), using 112 of them for
learning and keeping 38 hidden for testing. The inputs were `quiz_avg`, `assignments_avg` and `attendance_pct`; the target was `score`.
Name, gender, district and program were **not** used as inputs.

**3. Performance (on the 38 unseen students).** MAE is about 3.3 points: on average the prediction is about 3 marks away from the real
score. R² is about 0.89: the three inputs explain roughly 89% of the differences in scores. The quiz average matters most
(about +0.58 exam points per extra quiz point). Caution: the test group is small, so these numbers are approximate.

**4. Risks & responsible use.**
- *Who could be harmed:* a student wrongly flagged "at risk" (a false alarm) may lose confidence or be treated differently; a student who is
  at risk but not flagged may miss help. In our test, 3 of 34 passing students were wrongly flagged and 1 of 4 failing students was missed.
- *Hidden unfairness:* attendance can reflect transport, health or money problems rather than effort, and the model learned from a single small cohort.
- *Use it to:* start a supportive conversation, offer tutoring, and as one input among others, with a human making the decision.
- *Do NOT use it to:* grade, rank or punish students, decide scholarships or admission, or show labels publicly. Share predictions only with the student and
  their adviser, and do not keep labels longer than needed.